# Module 3: TESS Light Curves

This module retrieves TESS light curve data for all stars.

In [ ]:
import streamlit as st
import pandas as pd
import sys
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Add src to path
sys.path.append(os.path.join(os.path.dirname(''), '..', 'src'))

st.set_page_config(
    page_title="Module 3: TESS Light Curves",
    page_icon="📈",
    layout="wide",
    initial_sidebar_state="collapsed",
)

In [ ]:
# Load CSS
def local_css(file_name):
    with open(file_name) as f:
        st.markdown(f'<style>{f.read()}</style>', unsafe_allow_html=True)

local_css("streamlit_app/static/style.css")

In [ ]:
# Page header
st.markdown("""
<div style="text-align: center; margin-top: 0.5rem; margin-bottom: 1rem;">
    <div style="font-size: 3rem; line-height: 1;">📈</div>
    <h1 style="margin: 0.25rem 0 0 0; font-weight: 700; font-size: 1.5rem;">
        Module 3: TESS Light Curves
    </h1>
</div>
""", unsafe_allow_html=True)

st.markdown("---")

In [ ]:
# Import module logic
from modules.module4_tess_lightcurves import TESSLightCurveModule
from modules.integrity_tracker import IntegrityTracker

from workspace import current_user, get_store, normalize_user_id

In [ ]:
# Import module logic
from modules.module4_tess_lightcurves import TESSLightCurveModule
from modules.integrity_tracker import IntegrityTracker

In [ ]:
# Load data from workspace
st.markdown("### Load Data")

user = current_user()
if user:
    store = get_store()
    uid = normalize_user_id(user)
    
    workspace_dir = os.path.join("data", "users", uid)
    
    if os.path.exists(workspace_dir):
        saved_files = [f for f in os.listdir(workspace_dir) if f.endswith('.csv')]
        
        if saved_files:
            selected_file = st.selectbox("Select saved dataset", saved_files)
            
            if st.button("📂 Load Dataset"):
                file_path = os.path.join(workspace_dir, selected_file)
                df = pd.read_csv(file_path)
                st.success(f"Loaded {len(df)} coordinates from {selected_file}")
                st.dataframe(df.head(10), use_container_width=True)
                st.session_state['loaded_data'] = df
        else:
            st.caption("No saved datasets found.")
    else:
        st.caption("No saved datasets found.")
else:
    st.caption("Sign in to load your saved datasets.")

In [ ]:
# Integrity check
if 'loaded_data' in st.session_state:
    df = st.session_state['loaded_data']
    
    tracker = IntegrityTracker()
    valid, error_msg = tracker.validate_input_for_module(df, 3)
    
    if not valid:
        st.error(error_msg)
        st.stop()

In [ ]:
# Run Module 3
if 'loaded_data' in st.session_state:
    st.markdown("---")
    
    col1, col2 = st.columns([1, 3])
    with col1:
        run_module = st.button("▶️ Run Module 3", type="primary")
    with col2:
        st.caption("Retrieve TESS light curve data")
    
    if run_module:
        module3 = TESSLightCurveModule()
        df, report = module3.retrieve_lightcurves(st.session_state['loaded_data'], use_mock=True)
        
        # Mark as Module 3 complete
        df = tracker.mark_module_complete(df, 3)
        
        # Display results
        summary = module3.get_success_summary()
        st.success(summary)
        
        st.subheader("Results")
        st.dataframe(df.head(20), use_container_width=True)
        
        st.session_state['module3_results'] = df

In [ ]:
# Save to workspace
if 'module3_results' in st.session_state:
    st.markdown("---")
    st.markdown("### Save to Workspace")
    
    if user:
        dataset_name = st.text_input(
            "Dataset name",
            value=f"module3_results_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}"
        )
        
        if st.button("💾 Save to Workspace"):
            try:
                file_path = os.path.join(workspace_dir, f"{dataset_name}.csv")
                st.session_state['module3_results'].to_csv(file_path, index=False)
                st.success(f"Saved {len(st.session_state['module3_results'])} coordinates as '{dataset_name}.csv'")
            except Exception as exc:
                st.error(f"Failed to save: {exc}")

In [ ]:
# Navigation
st.markdown("---")
st.markdown("### Next Steps")

col1, col2 = st.columns(2)
with col1:
    if st.button("🎯 Go to Module 4"):
        st.info("Opening Module 4: Transit Detection...")
with col2:
    if st.button("🏠 Return to Navigation Hub"):
        st.info("Returning to Navigation Hub...")